# Analyzing Multistate Trajectory Files with openfe-analysis

This notebook demonstrates how to load, align, and analyze multistate
trajectory files produced by OpenFE's free energy protocols. We cover
two protocols:

- **Relative Hybrid Topology Protocol (RBFE)**: a single hybrid ligand is used
  to represent both end states. The trajectory contains one ligand
  per frame.
- **Separated Topologies Protocol (SepTop)**: both ligands are present
  simultaneously throughout the simulation. The trajectory contains
  two physically distinct ligands per frame.

Both protocols produce complex (protein + ligand) and solvent
(ligand only) legs, which will be analyzed separately.

## Removal of periodic boundary conditions and alignment operations

Before computing any RMSD-based metric we must:

1. **Unwrap** molecules that have been split across periodic boundary
   conditions (PBC), making each molecule whole again.
2. **Image** molecules into the same periodic image — in a complex
   simulation the ligand may drift into a neighbouring box relative
   to the protein. In SepTop, each ligand must be imaged
   *independently*, since they can cross periodic boundaries at
   different times.
3. **Align** all frames to a common reference (the first frame) by
   minimizing the protein Cα RMSD. This removes overall translational
   and rotational motion so that structural changes reflect internal
   dynamics rather than rigid-body drift.

`openfe-analysis` handles all of this automatically through its
trajectory transformation pipeline.

## Download tutorial data

In [ ]:
%%bash

# Hybrid topology
wget --show-progress "https://zenodo.org/records/20442933/files/openfe_analysis_skipped.tar.gz" -nc
tar -xzf openfe_analysis_skipped.tar.gz

# SepTop
wget --show-progress "https://zenodo.org/records/20507106/files/septop_structural_results.zip" -nc
tar -xzf septop_structural_results.zip

## Imports and helper functions

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import MDAnalysis as mda
import netCDF4 as nc
import numpy as np
from rdkit import Chem
from rdkit.Chem import SDMolSupplier

from openfe_analysis.rmsd import (
    LigandCOMDrift,
    Protein2DRMSD,
    RMSDAnalysis,
    SymmetryCorrectedLigandRMSD,
    gather_rms_data,
)
from openfe_analysis.utils.apply_transformations import (
    apply_complex_alignment_transformations,
    apply_ligand_alignment_transformations,
)
from openfe_analysis.utils.universe_utils import create_universe_single_state

In [ ]:
def compute_frame_stride(ds):
    """
    Compute a frame stride that gives approximately 500 analyzed frames
    per state, accounting for PositionInterval if present.

    Parameters
    ----------
    ds : netCDF4.Dataset
        Open NetCDF dataset for the multistate trajectory.

    Returns
    -------
    skip : int
        Frame stride to use in analysis ``.run(step=skip)`` calls.
    n_frames : int
        Total number of position frames available per state.
    """
    if hasattr(ds, "PositionInterval"):
        n_frames = len(
            range(0, ds.dimensions["iteration"].size, ds.PositionInterval)
        )
    else:
        n_frames = ds.dimensions["iteration"].size
    skip = max(n_frames // 500, 1)
    return skip, n_frames

In [ ]:
rfe_dir = pathlib.Path("openfe_analysis_skipped")
septop_dir = pathlib.Path("septop_structural_results")

# Part 1 — Relative Hybrid Topology Protocol

## 1.1 Complex leg

### Load trajectory and topology

In [ ]:
rfe_complex_pdb = rfe_dir / "hybrid_system.pdb"
rfe_complex_nc = rfe_dir / "simulation.nc"

### Compute structural metrics across all lambda states

We loop over all lambda states, loading each as a separate MDAnalysis
Universe.

In [ ]:
rfe_lig_rmsd = []
rfe_lig_com_drift = []
rfe_prot_2d_rmsd = []
rfe_time_ps = None
    
protein_selection = "protein and name CA"
ligand_selection = "resname UNK"
with nc.Dataset(rfe_complex_nc) as ds:
    n_lambda = ds.dimensions["state"].size

    # Account for the position skip rate.
    if hasattr(ds, "PositionInterval"):
        n_frames = len(range(0, ds.dimensions["iteration"].size, ds.PositionInterval))
    else:
        n_frames = ds.dimensions["iteration"].size

    # find skip that would give ~500 frames of output
    # max against 1 to avoid skip=0 case
    skip = max(n_frames // 500, 1)

    u_top = mda.Universe(rfe_complex_pdb)
    
    # Loop over all lambda states
    for state_idx in range(n_lambda):
        # Create an MDAnalysis universe for this state
        universe = create_universe_single_state(u_top._topology, ds, state_idx)
        prot = universe.select_atoms(protein_selection)
        lig = universe.select_atoms(ligand_selection)
        
        # Remove PBC artifacts and align frames
        apply_complex_alignment_transformations(universe, protein=prot, ligands=[lig])

        # Calculate the protein 2D RMSE
        protein_2d_rmsd = Protein2DRMSD(prot).run(step=skip)
        rfe_prot_2d_rmsd.append(protein_2d_rmsd.results.rmsd2d)

        # Calculate the ligand RMSD
        ligand_rmsd = RMSDAnalysis(lig, mass_weighted=True).run(step=skip).results.rmsd
        rfe_lig_rmsd.append(ligand_rmsd)

        # Calculate the Ligand COM drift
        ligand_com_drift = LigandCOMDrift(lig).run(step=skip).results.com_drift
        rfe_lig_com_drift.append(ligand_com_drift)
        
        # Get the time frames
        if rfe_time_ps is None:
            rfe_time_ps = (
                np.arange(len(universe.trajectory))[::skip] * universe.trajectory.dt
            )

### Plot results

Each line represents one lambda state. Consistent profiles across
states indicate a stable, well-behaved simulation. Large deviations
in specific states may indicate sampling problems or instabilities.

In [ ]:
from openfe_analysis.utils.plotting import (
    plot_2D_rmsd,
    plot_ligand_RMSD,
    plot_ligand_COM_drift,
)

In [ ]:
# Protein 2D RMSD — all states in one figure
fig = plot_2D_rmsd(rfe_prot_2d_rmsd)
plt.show()

In [ ]:
# Ligand RMSD
fig = plot_ligand_RMSD(rfe_time_ps, rfe_lig_rmsd)
plt.show()

In [ ]:
# Ligand COM drift
fig = plot_ligand_COM_drift(rfe_time_ps, rfe_lig_com_drift)
plt.show()

# Part 2 — Separated Topologies (SepTop)

In a SepTop simulation both ligands are present simultaneously
throughout the calculation. Ligand A is coupled at the start of
the lambda schedule while ligand B is decoupled, and the schedule
then decouples A and couples B. This means:

- Both ligands must be aligned and imaged for every lambda state.
- Each ligand must be imaged **independently** relative to the protein.
  If the two ligands were treated as a single combined AtomGroup,
  `ClosestImageShift` would shift them based on their combined centre
  of mass, which can fail to correct individual ligand jumps across
  periodic boundaries.
- `SymmetryCorrectedLigandRMSD` is used for both ligands since the
  SepTop setup preserves the full atom identity of each molecule.

## 2.1 SepTop Complex leg

### Load trajectory and topology

In [ ]:
septop_complex_pdb = septop_dir / "complex" / "alchemical_system.pdb"
septop_complex_nc_file = septop_dir / "complex" / "complex.nc"

rdmol_A = SDMolSupplier(str(septop_dir / "complex" / "ligand_A.sdf"), removeHs=False)[0]
rdmol_B = SDMolSupplier(str(septop_dir / "complex" / "ligand_B.sdf"), removeHs=False)[0]

### Getting ligand indices

In a real SepTop calculation, the ligand indices are stored in
the protocol output JSON and can be extracted from there. For this
tutorial we derive them directly from the topology by residue name, which
works because the two ligands have distinct residue indices within the
same residue name `UNK`.

In [ ]:
u_tmp = mda.Universe(septop_complex_pdb)

# Select all UNK residues — in a SepTop system these are the two ligands
unk_residues = u_tmp.select_atoms("resname UNK").residues

septop_ligand_A_indices = unk_residues[0].atoms.indices.tolist()
septop_ligand_B_indices = unk_residues[1].atoms.indices.tolist()

### Compute structural metrics across all lambda states

In [ ]:
septop_prot_rmsds = []
septop_lig_A_rmsds = []
septop_lig_B_rmsds = []
septop_lig_A_drifts = []
septop_lig_B_drifts = []
septop_prot_2d_rmsds = []
septop_time_ps = None

with nc.Dataset(septop_complex_nc_file) as ds:
    n_lambda = ds.dimensions["state"].size

    # Account for the position skip rate.
    if hasattr(ds, "PositionInterval"):
        n_frames = len(range(0, ds.dimensions["iteration"].size, ds.PositionInterval))
    else:
        n_frames = ds.dimensions["iteration"].size

    # find skip that would give ~500 frames of output
    # max against 1 to avoid skip=0 case
    septop_skip = max(n_frames // 500, 1)

    u_top = mda.Universe(septop_complex_pdb)
    
    for state_idx in range(n_lambda):
        
        # Create an MDAnalysis universe for this state
        universe = create_universe_single_state(u_top._topology, ds, state_idx)
        prot = universe.select_atoms(protein_selection)
        lig_A = universe.atoms[septop_ligand_A_indices]
        lig_B = universe.atoms[septop_ligand_B_indices]

        # Remove PBC artifacts and align frames
        apply_complex_alignment_transformations(universe, protein=prot, ligands=[lig_A, lig_B])

        # Calculate the protein 2D RMSD
        septop_prot_2d_rmsds.append(
            Protein2DRMSD(prot).run(step=septop_skip).results.rmsd2d
        )

        # Calculate the symmetry corrected RMSD
        septop_lig_A_rmsds.append(
            SymmetryCorrectedLigandRMSD(lig_A, rdmol=rdmol_A)
            .run(step=septop_skip).results.rmsd
        )
        septop_lig_B_rmsds.append(
            SymmetryCorrectedLigandRMSD(lig_B, rdmol=rdmol_B)
            .run(step=septop_skip).results.rmsd
        )
        # Calculate the Ligand COM drift
        septop_lig_A_drifts.append(
            LigandCOMDrift(lig_A).run(step=septop_skip).results.com_drift
        )
        septop_lig_B_drifts.append(
            LigandCOMDrift(lig_B).run(step=septop_skip).results.com_drift
        )
        

        if septop_time_ps is None:
            septop_time_ps = (
                np.arange(len(universe.trajectory))[::septop_skip] * universe.trajectory.dt
            )

In [ ]:
# Protein 2D RMSD — all states in one figure
fig = plot_2D_rmsd(septop_prot_2d_rmsds)
plt.show()

In [ ]:
# Ligand A RMSD
fig = plot_ligand_RMSD(septop_time_ps, septop_lig_A_rmsds)
fig.suptitle("Ligand A RMSD")
plt.show()

In [ ]:
# Ligand B RMSD
fig = plot_ligand_RMSD(septop_time_ps, septop_lig_B_rmsds)
fig.suptitle("Ligand B RMSD")
plt.show()

In [ ]:
# Ligand A COM drift
fig = plot_ligand_COM_drift(septop_time_ps, septop_lig_A_drifts)
fig.suptitle("Ligand A COM drift")
plt.show()

In [ ]:
# Ligand B COM drift
fig = plot_ligand_COM_drift(septop_time_ps, septop_lig_B_drifts)
fig.suptitle("Ligand B COM drift")
plt.show()